# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their fields, and corresponding @ids
record_sets = metadata.record_sets  # This returns mlcroissant's RecordSet objects

if not record_sets:
    print('No record sets found in the metadata!')
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}, @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                print(f"  - Field: {field.name}, @id: {field.id}, datatype: {getattr(field, 'data_type', 'unknown')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record_set @ids
record_set_ids = [rs.id for rs in record_sets]

# Extract all record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set {record_set_id} with {len(records)} records, columns: {dataframes[record_set_id].columns.tolist()}")
    print()

# For demonstration, choose the first record set as the main set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f"Main record set columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print('No record set dataframes to display.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# If there is a typical numeric field (e.g., 'Age'), use its @id. Otherwise, choose any numeric field found.

numeric_field = None
group_field = None
df = None

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to find some numeric-typed field
    for rs in record_sets:
        if rs.id == main_record_set_id:
            for field in rs.fields:
                if getattr(field, 'data_type', '').lower() in ['integer', 'float', 'number']:
                    numeric_field = field.id
                    break
            # For grouping, try to take the first non-numeric field
            for field in rs.fields:
                if getattr(field, 'data_type', '').lower() not in ['integer', 'float', 'number']:
                    group_field = field.id
                    break
            break

    if numeric_field and numeric_field in df.columns:
        print(f"Using numeric field: {numeric_field}")
        threshold = 50  # You may adjust threshold based on the column statistics
        # Ensure no NAs and try converting to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print('No suitable group_field found for grouping.')
    else:
        print("No numeric field detected in main record set for analysis.")
else:
    print('No main record set found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualization of the main numeric field, if available
if df is not None and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optional: if a grouping field is available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('Not enough numeric data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset was loaded using its Croissant schema with the `mlcroissant` library.
- We reviewed the available record sets and their fields, identified and explored numeric attributes.
- Filtered, normalized, and grouped data as part of EDA.
- Visualized the main numeric distribution and group differences (when possible).

Use this notebook as a starting point for more domain-specific analyses or modeling with your clinical oncology dataset.